# Liquid Clustering

## Overview

Liquid Clustering is a **modern data organization technique** that provides a flexible way to organize data based on the columns most frequently used in queries. Unlike traditional partitioning, it does not require you to create separate partitions for each value of a specific column — instead, Databricks intelligently clusters the data for you.

## Key Advantages

- **Adaptable clustering columns:** You can change the clustering columns whenever your query patterns change, without recreating the entire table.
- **Better data skipping:** Liquid Clustering provides a flexible approach to organizing data, which improves data skipping and query performance.
- **No partition maintenance overhead:** Avoids the pitfalls of traditional partitioning, such as over-partitioning (small files problem) and under-partitioning (large scans).

## Summary

| Aspect | Traditional Partitioning | Liquid Clustering |
|---|---|---|
| Data organization | Static partitions per column value | Dynamic clustering on chosen columns |
| Changing layout | Requires table recreation | Re-cluster by altering the table |
| Data skipping | Limited | Optimized |

> **Note:** Because clustering keys can be updated as query patterns evolve, Liquid Clustering is the recommended replacement for partitioning and Z-ordering on Delta tables in Databricks.

# Liquid Clustering — Interview Answer

**Q: What is Liquid Clustering in Databricks, and why use it over partitioning or Z-ordering?**

**A:** Liquid Clustering is Databricks' modern technique for organizing data in Delta tables based on the columns that are most frequently filtered in queries. Instead of physically partitioning data by a column's values (like traditional partitioning) or rewriting files based on Z-order, Databricks dynamically maintains a clustering layout behind the scenes.

**Key points to mention in an interview:**

1. **How it works:** You declare clustering keys when creating the table (e.g., `CLUSTER BY (customer_id, order_date)`), and Databricks incrementally reorganizes data as it's written — no full table rewrite needed.

2. **Adaptable keys:** Unlike partitioning or Z-ordering, you can change the clustering columns later (`ALTER TABLE ... CLUSTER BY (...)`) as query patterns evolve — without recreating the table.

3. **Better data layout:** It replaces both Hive-style partitioning and Z-ordering (OPTIMIZE ZORDER BY), delivering improved data skipping and query performance while avoiding the small-files problem caused by over-partitioning.

4. **Incremental maintenance:** Unlike Z-ordering, which must rewrite all files each time OPTIMIZE runs, liquid clustering re-clusters only where it's beneficial.

**Sample syntax:**

```sql
CREATE TABLE sales (
  order_id BIGINT,
  customer_id STRING,
  order_date DATE
)
CLUSTER BY (customer_id, order_date);

-- Change clustering keys later as query patterns evolve:
ALTER TABLE sales CLUSTER BY (order_date);

-- Incrementally re-cluster existing data:
OPTIMIZE sales;
```

**One-liner summary:** Liquid Clustering is the recommended replacement for partitioning and Z-ordering on Delta tables because it gives flexible, adaptive data organization with better data skipping and no partition maintenance overhead.


In [0]:
# Practical Example: Liquid Clustering in Action

# 1. Create a table with liquid clustering keys declared at creation time
spark.sql("""
CREATE OR REPLACE TABLE demo_sales (
  order_id     BIGINT,
  customer_id  STRING,
  order_date   DATE,
  amount       DOUBLE
)
CLUSTER BY (customer_id, order_date)
""")

# 2. Insert sample data — Databricks incrementally organizes it as it is written
spark.sql("""
INSERT INTO demo_sales
SELECT
  id AS order_id,
  CONCAT('cust_', CAST(id % 50 AS STRING)) AS customer_id,  -- 50 customers
  date_add(DATE'2024-01-01', CAST(id % 90 AS INT)) AS order_date,
  RAND(42) * 1000 AS amount
FROM range(100000)
""")

# 3. Verify the clustering keys on the table
spark.sql("DESCRIBE DETAIL demo_sales").select("name", "clusteringColumns").display()

# 4. Query the table on the clustered columns — data skipping kicks in
spark.sql("""
SELECT customer_id, COUNT(*) AS orders, SUM(amount) AS total
FROM demo_sales
WHERE customer_id = 'cust_7' AND order_date BETWEEN '2024-02-01' AND '2024-02-28'
GROUP BY customer_id
""").display()

# 5. Query patterns changed? Swap clustering keys — no table recreation needed
spark.sql("ALTER TABLE demo_sales CLUSTER BY (order_date)")

# 6. Incrementally re-cluster existing data (only rewrites where beneficial,
#    unlike ZORDER BY which rewrites everything)
spark.sql("OPTIMIZE demo_sales")

spark.sql("DESCRIBE DETAIL demo_sales").select("name", "clusteringColumns").display()